In [6]:
import pandas as pd 
import os
import json
import re
import shutil
from matplotlib import pyplot as plt
import numpy as np
import hashlib

In [7]:
def add_sha1_column(df, path_column='Image Path', sha1_column='SHA1'):
    def compute_sha1(filepath):
        try:
            with open(filepath, 'rb') as f:
                return hashlib.sha1(f.read()).hexdigest()
        except Exception as e:
            return None

    df = df.copy()
    df[sha1_column] = df[path_column].apply(compute_sha1)
    return df

In [8]:
def get_duplicate_images(df, sha1_column='SHA1'):
    # Count SHA1 frequencies
    sha1_counts = df[sha1_column].value_counts()
    duplicate_hashes = sha1_counts[sha1_counts > 1].index

    # Filter rows where SHA1 is in the duplicate list
    return df[df[sha1_column].isin(duplicate_hashes)].sort_values(by=sha1_column)

In [9]:
def test_image_paths(df):
    for i in range(len(df)):
        plt.imshow(plt.imread(df["Image Path"].loc[i]))
        plt.close()

## Load Dataset into DataFrame

In [10]:
dataset_df = pd.read_csv('co-mof_dataset/mof_dataset.csv')

## Load Synthesis Conditions into Python Dictionary

In [11]:
with open('co-mof_dataset/synthesis_conditions.json') as f:
    synth_conds = json.load(f)
    
synth_cond_counts = {cond: 0 for cond in synth_conds}

for i in range(len(dataset_df)):
    synth_cond_counts[dataset_df["Synthesis ID"].iloc[i]] += 1

### Attempt to open each Image Path in Dataset (will throw an exception if there are any bad paths)

In [12]:
test_image_paths(dataset_df)

In [13]:
print(f'There are {len(dataset_df)} images in the dataset')
print(f'There are {len(synth_conds)} synthesis conditions')

There are 787 images in the dataset
There are 50 synthesis conditions


In [14]:
print('Synthesis Condition Counts:')
for c in synth_cond_counts:
    print(f'{c}: {synth_cond_counts[c]}')

Synthesis Condition Counts:
SolventVolumes-1I: 4
SolventVolumes-2I: 5
SolventVolumes-3I: 4
SolventVolumes-4I: 6
SolventVolumes-5I: 18
SolventVolumes-6I: 15
Time1: 3
Time2: 7
Time3: 8
Time4: 15
Time5: 6
Time6: 11
Temperature1: 2
Temperature2: 1
Temperature4: 4
Temperature5: 10
SolventVolumes-1II: 2
SolventVolumes-2II: 21
SolventVolumes-3II: 51
SolventVolumes-4II: 64
SolventVolumes-5II: 35
SolventVolumes-6II: 24
SolventVolumes-7II: 2
SolventVolumes-8II: 2
SolventVolumes-9II: 4
SolventVolumes-10II: 15
SolventVolumes-11II: 54
SolventVolumes-12II: 28
SolventVolumes-13II: 56
SolventVolumes-14II: 41
SolventVolumes-15II: 2
SolventVolumes-16II: 1
SolventVolumes-17II: 2
SolventVolumes-18II: 2
SolventVolumes-19II: 26
SolventVolumes-20II: 19
SolventVolumes-21II: 40
SolventVolumes-22II: 40
SolventVolumes-23II: 41
Modulator-Linker1: 2
Modulator-Linker2: 9
Modulator-Linker3: 5
Modulator-Linker4: 4
Modulator-Linker5: 5
Modulator-Linker6: 13
Modulator-Linker7: 22
Modulator-Linker8: 15
Modulator-Linker9

In [15]:
print('Unused Synthesis Conditions:')
for c in synth_cond_counts:
    if synth_cond_counts[c] == 0:
        print(c)

Unused Synthesis Conditions:


## Look for duplicate image entries

In [16]:
dataset_df = add_sha1_column(dataset_df)

In [17]:
duplicates_df = get_duplicate_images(dataset_df)

In [18]:
print(f'Number of duplicate files: {len(duplicates_df)}')
print(f'Number of unique images: {len(dataset_df["SHA1"].unique())}')

Number of duplicate files: 0
Number of unique images: 787


In [19]:
if len(duplicates_df) > 0:
    duplicates_df.to_csv('co-mof_dataset/duplicates.csv')